# YOLO Defect Detection — Training & Visualization

This notebook lets you:
1. **Configure** training hyperparameters
2. **Run training** with live metric updates
3. **Visualize** loss and mAP curves after each epoch
4. **Inspect** predictions on validation images

## 0. Setup

In [1]:
import sys
from pathlib import Path

# Make sure we run from the repo root regardless of where the notebook is
REPO_ROOT = Path("__file__").resolve().parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from IPython.display import display, clear_output
import ipywidgets as widgets
from ultralytics import YOLO
from ultralytics.utils.plotting import Annotator, colors
import cv2
from PIL import Image
import torch

print(f"PyTorch  : {torch.__version__}")
print(f"MPS      : {torch.backends.mps.is_available()}")
print(f"CUDA     : {torch.cuda.is_available()}")

Matplotlib is building the font cache; this may take a moment.


PyTorch  : 2.10.0
MPS      : True
CUDA     : False


## 1. Training Configuration

In [2]:
# ── Edit these before running ────────────────────────────────────────────────
MODEL    = "yolo11s.pt"                        # pretrained weights to fine-tune
DATA     = "../configs/pcb_detection.yaml"  # dataset config
EPOCHS   = 50
IMGSZ    = 640
BATCH    = 16                                  # -1 = auto
DEVICE   = "mps"                               # 'mps', 'cpu', '0' (GPU index)
RUN_NAME = "pcb_detection"
PROJECT  = "../runs/train"
# ─────────────────────────────────────────────────────────────────────────────

print("Config ready. Run the next cell to start training.")

Config ready. Run the next cell to start training.


## 2. Train with Live Visualization

Metrics are plotted **after every epoch** using a custom Ultralytics callback.

In [4]:
# ── Live-plot state ───────────────────────────────────────────────────────────
history = {
    "epoch":        [],
    "train/box_loss": [], "train/cls_loss": [], "train/dfl_loss": [],
    "val/box_loss":   [], "val/cls_loss":   [], "val/dfl_loss":   [],
    "metrics/mAP50":  [], "metrics/mAP50-95": [],
    "metrics/precision": [], "metrics/recall": [],
}

plot_output = widgets.Output()
display(plot_output)

def _update_history(trainer):
    """Callback: collect metrics logged by Ultralytics after each epoch."""
    m = trainer.metrics          # dict logged by the trainer
    lr_metrics = getattr(trainer, 'lr', {})
    
    history["epoch"].append(trainer.epoch + 1)
    
    # losses are stored on the trainer loss_items
    loss_names = getattr(trainer, 'loss_names', [])
    loss_items = getattr(trainer, 'loss_items', [])
    tloss       = getattr(trainer, 'tloss',      None)

    # Map YOLO's standard loss names
    loss_map = {}
    if loss_names and loss_items is not None:
        for name, val in zip(loss_names, loss_items):
            loss_map[name] = float(val)

    history["train/box_loss"].append(loss_map.get("box_loss", float("nan")))
    history["train/cls_loss"].append(loss_map.get("cls_loss", float("nan")))
    history["train/dfl_loss"].append(loss_map.get("dfl_loss", float("nan")))

    history["val/box_loss"].append(float(m.get("val/box_loss",   float("nan"))))
    history["val/cls_loss"].append(float(m.get("val/cls_loss",   float("nan"))))
    history["val/dfl_loss"].append(float(m.get("val/dfl_loss",   float("nan"))))

    history["metrics/mAP50"].append(float(m.get("metrics/mAP50",    float("nan"))))
    history["metrics/mAP50-95"].append(float(m.get("metrics/mAP50-95", float("nan"))))
    history["metrics/precision"].append(float(m.get("metrics/precision", float("nan"))))
    history["metrics/recall"].append(float(m.get("metrics/recall",    float("nan"))))


def _draw_live(trainer):
    """Callback: redraw the live charts."""
    if len(history["epoch"]) < 1:
        return

    epochs = history["epoch"]

    with plot_output:
        clear_output(wait=True)
        fig, axes = plt.subplots(2, 2, figsize=(13, 8))
        fig.suptitle(f"Training — Epoch {epochs[-1]} / {EPOCHS}", fontsize=13, fontweight="bold")

        # ── Train losses ──────────────────────────────────────────────────────
        ax = axes[0, 0]
        ax.plot(epochs, history["train/box_loss"], label="box",  color="#e74c3c")
        ax.plot(epochs, history["train/cls_loss"], label="class", color="#3498db")
        ax.plot(epochs, history["train/dfl_loss"], label="dfl",  color="#2ecc71")
        ax.set_title("Train Loss")
        ax.set_xlabel("Epoch")
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

        # ── Val losses ────────────────────────────────────────────────────────
        ax = axes[0, 1]
        ax.plot(epochs, history["val/box_loss"], label="box",  color="#e74c3c")
        ax.plot(epochs, history["val/cls_loss"], label="class", color="#3498db")
        ax.plot(epochs, history["val/dfl_loss"], label="dfl",  color="#2ecc71")
        ax.set_title("Validation Loss")
        ax.set_xlabel("Epoch")
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

        # ── mAP ───────────────────────────────────────────────────────────────
        ax = axes[1, 0]
        ax.plot(epochs, history["metrics/mAP50"],    label="mAP@50",    color="#9b59b6")
        ax.plot(epochs, history["metrics/mAP50-95"], label="mAP@50-95", color="#e67e22")
        ax.set_title("mAP")
        ax.set_xlabel("Epoch")
        ax.set_ylim(0, 1)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

        # ── Precision / Recall ────────────────────────────────────────────────
        ax = axes[1, 1]
        ax.plot(epochs, history["metrics/precision"], label="Precision", color="#1abc9c")
        ax.plot(epochs, history["metrics/recall"],    label="Recall",    color="#f39c12")
        ax.set_title("Precision & Recall")
        ax.set_xlabel("Epoch")
        ax.set_ylim(0, 1)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

        plt.tight_layout()
        plt.show()


# ── Run training ─────────────────────────────────────────────────────────────
model = YOLO(MODEL)
model.add_callback("on_train_epoch_end", _update_history)
model.add_callback("on_train_epoch_end", _draw_live)

results = model.train(
    data=DATA,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    project=PROJECT,
    name=RUN_NAME,
    verbose=False,   # suppress per-batch logs; charts replace them
)

BEST_WEIGHTS = Path(results.save_dir) / "weights" / "best.pt"
print(f"\nTraining done. Best weights → {BEST_WEIGHTS}")

Output()

Ultralytics 8.4.14 🚀 Python-3.13.1 torch-2.10.0 MPS (Apple M2)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../configs/pcb_detection.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=pcb_detection, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=T

## 3. Post-Training Analysis

> Run these cells after training completes (or point `BEST_WEIGHTS` at an existing run).

In [5]:
# If you're re-opening the notebook, set the path to your saved weights here:
# BEST_WEIGHTS = Path("../runs/train/defect_detection/weights/best.pt")

RUN_DIR = BEST_WEIGHTS.parent.parent
CSV_PATH = RUN_DIR / "results.csv"

print(f"Run dir : {RUN_DIR}")
print(f"Weights : {BEST_WEIGHTS}")
print(f"CSV     : {CSV_PATH}")

Run dir : /Users/manussanun/workspace/investic/poc-yolo/runs/runs/train/pcb_detection
Weights : /Users/manussanun/workspace/investic/poc-yolo/runs/runs/train/pcb_detection/weights/best.pt
CSV     : /Users/manussanun/workspace/investic/poc-yolo/runs/runs/train/pcb_detection/results.csv


### 3a. Full Training Curves (from `results.csv`)

In [6]:
df = pd.read_csv(CSV_PATH)
df.columns = df.columns.str.strip()  # strip whitespace from column names
print("Columns:", list(df.columns))
df.tail(5)

Columns: ['epoch', 'time', 'train/box_loss', 'train/cls_loss', 'train/dfl_loss', 'metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'val/box_loss', 'val/cls_loss', 'val/dfl_loss', 'lr/pg0', 'lr/pg1', 'lr/pg2']


,epoch,time,train/box_loss,train/cls_loss,train/dfl_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/dfl_loss,lr/pg0,lr/pg1,lr/pg2
45,46,2528.24,0.80035,0.38518,0.97613,0.99353,1.0,0.995,0.78042,0.91075,0.37513,1.10467,0.000218,0.000218,0.000218
46,47,2577.79,0.77687,0.39501,0.99032,0.99334,1.0,0.995,0.77874,0.91679,0.37735,1.10569,0.000178,0.000178,0.000178
47,48,2639.12,0.79265,0.37609,1.00228,0.99534,1.0,0.995,0.77789,0.90761,0.37315,1.10069,0.000139,0.000139,0.000139
48,49,2685.53,0.74014,0.35707,0.95496,0.99589,1.0,0.995,0.77604,0.90675,0.36341,1.09794,0.000099,0.000099,0.000099
49,50,2738.10,0.71481,0.34978,0.94363,0.99581,1.0,0.995,0.78170,0.90060,0.35457,1.09384,0.000060,0.000060,0.000060


In [7]:
def plot_training_curves(df):
    epochs = df["epoch"]

    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    fig.suptitle("Full Training History", fontsize=14, fontweight="bold")

    def _plot(ax, cols, title, ylabel="Loss", ylim=None):
        color_cycle = ["#e74c3c", "#3498db", "#2ecc71", "#9b59b6", "#e67e22"]
        for col, c in zip(cols, color_cycle):
            if col in df.columns:
                ax.plot(epochs, df[col], label=col.split("/")[-1], color=c)
        ax.set_title(title)
        ax.set_xlabel("Epoch")
        ax.set_ylabel(ylabel)
        if ylim:
            ax.set_ylim(*ylim)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

    _plot(axes[0,0], ["train/box_loss", "train/cls_loss", "train/dfl_loss"], "Train Loss")
    _plot(axes[0,1], ["val/box_loss",   "val/cls_loss",   "val/dfl_loss"],   "Val Loss")
    _plot(axes[0,2], ["metrics/mAP50", "metrics/mAP50-95"], "mAP", ylabel="mAP", ylim=(0,1))
    _plot(axes[1,0], ["metrics/precision", "metrics/recall"], "Precision & Recall", ylabel="Score", ylim=(0,1))

    # LR schedule
    lr_cols = [c for c in df.columns if c.startswith("lr/")]
    if lr_cols:
        _plot(axes[1,1], lr_cols, "Learning Rate", ylabel="LR")
    else:
        axes[1,1].set_visible(False)

    # Best epoch marker on mAP chart
    if "metrics/mAP50" in df.columns:
        best_epoch = df["metrics/mAP50"].idxmax()
        best_map   = df["metrics/mAP50"].max()
        axes[0,2].axvline(epochs[best_epoch], color="black", linestyle="--", alpha=0.5)
        axes[0,2].annotate(
            f"best\n{best_map:.3f}",
            xy=(epochs[best_epoch], best_map),
            xytext=(5, -20), textcoords="offset points",
            fontsize=8, color="black",
        )

    axes[1,2].set_visible(False)  # spare slot
    plt.tight_layout()
    plt.savefig(RUN_DIR / "training_curves.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {RUN_DIR}/training_curves.png")

plot_training_curves(df)

/var/folders/07/nc0nl4893tx3ls59ktfkrd8c0000gn/T/ipykernel_85652/3737009210.py:17: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend(fontsize=8)


<Figure size 1600x900 with 6 Axes>

Saved → /Users/manussanun/workspace/investic/poc-yolo/runs/runs/train/pcb_detection/training_curves.png


### 3b. YOLO-generated Plots (confusion matrix, PR curve, F1 curve)

In [8]:
auto_plots = [
    ("confusion_matrix.png",          "Confusion Matrix"),
    ("confusion_matrix_normalized.png","Confusion Matrix (normalized)"),
    ("PR_curve.png",                   "Precision-Recall Curve"),
    ("F1_curve.png",                   "F1 Curve"),
    ("R_curve.png",                    "Recall Curve"),
    ("P_curve.png",                    "Precision Curve"),
    ("labels.jpg",                     "Label Distribution"),
]

found = [(p, t) for p, t in auto_plots if (RUN_DIR / p).exists()]

if not found:
    print("No auto-generated plots found yet — run training first.")
else:
    cols = 3
    rows = (len(found) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 5 * rows))
    axes = np.array(axes).flatten()

    for ax, (fname, title) in zip(axes, found):
        img = np.array(Image.open(RUN_DIR / fname))
        ax.imshow(img)
        ax.set_title(title, fontsize=10)
        ax.axis("off")

    for ax in axes[len(found):]:
        ax.set_visible(False)

    plt.tight_layout()
    plt.show()

<Figure size 1800x500 with 3 Axes>

### 3c. Sample Validation Predictions

In [ ]:
def show_val_predictions(weights_path, data_yaml, n=8, conf=0.25, imgsz=640):
    """Run the best model on validation images and display annotated results."""
    import yaml
    
    with open(data_yaml) as f:
        cfg = yaml.safe_load(f)
    
    data_root = Path(data_yaml).parent.parent / cfg["path"]
    val_dir   = data_root / cfg["val"]
    
    img_paths = sorted(val_dir.glob("*.*"))[:n]
    if not img_paths:
        print(f"No validation images found in {val_dir}")
        return

    model = YOLO(weights_path)
    results = model.predict(img_paths, conf=conf, imgsz=imgsz, verbose=False)

    cols = 4
    rows = (len(results) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
    axes = np.array(axes).flatten()

    for ax, r in zip(axes, results):
        annotated = r.plot()          # BGR numpy array with boxes drawn
        ax.imshow(annotated[..., ::-1])  # convert BGR→RGB
        n_det = len(r.boxes)
        ax.set_title(f"{Path(r.path).name}\n{n_det} detection(s)", fontsize=8)
        ax.axis("off")

    for ax in axes[len(results):]:
        ax.set_visible(False)

    plt.suptitle(f"Validation Predictions (conf ≥ {conf})", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()


show_val_predictions(BEST_WEIGHTS, DATA, n=8, conf=0.25)

### 3d. Per-class mAP Summary

In [ ]:
model  = YOLO(BEST_WEIGHTS)
metrics = model.val(data=DATA, verbose=False)

class_names = list(model.names.values())
ap50 = metrics.box.ap50         # per-class AP@50
ap   = metrics.box.ap           # per-class AP@50-95

summary = pd.DataFrame({
    "Class":    class_names,
    "AP@50":    ap50.tolist(),
    "AP@50-95": ap.tolist(),
})

# Add overall row
summary.loc[len(summary)] = ["mean (all)", metrics.box.map50, metrics.box.map]
print(summary.to_string(index=False, float_format="{:.4f}".format))

# Bar chart
fig, ax = plt.subplots(figsize=(max(6, len(class_names) * 1.5), 4))
x = np.arange(len(class_names))
w = 0.35
ax.bar(x - w/2, ap50,  w, label="AP@50",    color="#9b59b6")
ax.bar(x + w/2, ap,    w, label="AP@50-95", color="#e67e22")
ax.set_xticks(x)
ax.set_xticklabels(class_names)
ax.set_ylabel("Average Precision")
ax.set_ylim(0, 1)
ax.set_title("Per-class AP")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()